# Report figures

Generates the four figures used in REPORT.md from the recorded results (pure
matplotlib — no data/GPU needed). Saves PNGs to the working directory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
OUT = "/kaggle/working"  # change to "." if running outside Kaggle
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

In [ ]:
# 1) Model ladder (NDCG@10)
labels = ["MostPop", "ItemKNN-cos", "ItemKNN-tfidf", "ItemKNN-bm25", "SASRec", "Hybrid (late)"]
vals   = [0.0171, 0.0418, 0.0451, 0.0709, 0.0735, 0.0798]
colors = ["#bbb", "#bbb", "#bbb", "#7aa6c2", "#2c7fb8", "#d95f0e"]
fig, ax = plt.subplots(figsize=(7, 3.6))
b = ax.barh(labels, vals, color=colors)
ax.bar_label(b, fmt="%.4f", padding=3, fontsize=9)
ax.set_xlabel("NDCG@10"); ax.set_xlim(0, 0.092)
ax.set_title("Model ladder — Yambda-50M (our harness)")
ax.invert_yaxis(); fig.tight_layout(); fig.savefig(f"{OUT}/fig1_ladder.png"); plt.show()

In [ ]:
# 2) Fusion weight curve (validation NDCG@10)
betas = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
ndcg  = [0.0702, 0.0734, 0.0751, 0.0784, 0.0788, 0.0809, 0.0812, 0.0807, 0.0794, 0.0766]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(betas, ndcg, "-o", color="#d95f0e")
ax.axhline(ndcg[0], ls="--", color="#2c7fb8", label=f"SASRec (β=0) = {ndcg[0]:.4f}")
ax.scatter([0.75], [0.0812], s=130, facecolors="none", edgecolors="k", zorder=5, label="peak β=0.75")
ax.set_xlabel("fusion weight β"); ax.set_ylabel("validation NDCG@10")
ax.set_title("Late fusion: content weight β (inverted-U, peak ≈ 0.75)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig2_beta.png"); plt.show()

In [ ]:
# 3) Robustness over 5 user splits
seeds  = [0, 1, 2, 3, 4]
sasrec = [0.0712, 0.0714, 0.0759, 0.0737, 0.0718]
fused  = [0.0776, 0.0802, 0.0809, 0.0804, 0.0801]
x = np.arange(len(seeds)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(x - w/2, sasrec, w, label="SASRec", color="#2c7fb8")
ax.bar(x + w/2, fused, w, label="Fused", color="#d95f0e")
for i in x:
    ax.text(i + w/2, fused[i] + 0.0008, f"+{100*(fused[i]-sasrec[i])/sasrec[i]:.0f}%", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([f"split {s}" for s in seeds])
ax.set_ylabel("test NDCG@10"); ax.set_ylim(0, 0.095)
ax.set_title("Robustness: +9.7% ± 2.0% across 5 held-out splits")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig3_robustness.png"); plt.show()

In [ ]:
# 4) Head vs tail lift (where content helps)
groups = ["≤5 / >5", "≤20 / >20", "≤100 / >100"]
tail_lift = [-8.5, 4.1, 7.9]
head_lift = [12.5, 12.9, 13.2]
x = np.arange(len(groups)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(x - w/2, tail_lift, w, label="tail items", color="#cc4c3b")
ax.bar(x + w/2, head_lift, w, label="head items", color="#3b8c4c")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_xlabel("popularity threshold (train interactions)")
ax.set_ylabel("NDCG@10 lift, fused vs SASRec (%)")
ax.set_title("Content helps the head, not the tail")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig4_head_tail.png"); plt.show()
print("saved: fig1_ladder.png, fig2_beta.png, fig3_robustness.png, fig4_head_tail.png")